# Functions and Mappings

| Difficulty | █░░░░░░░░░ 1/10 |
| :--- | :--- |
| Prerequisites | 00.01, 00.02 |
| Time | ~40 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/00_Prerequisites/03_functions_and_mappings.ipynb)

---

## 🎯 Learning Objective

Understand functions as mappings between sets — domain, range, injectivity/surjectivity/bijectivity, composition, and inverses — the vocabulary used to describe every layer of a neural network.

---

## 📐 Theory

A **function** $f: X \to Y$ assigns exactly one output in $Y$ to every input in $X$. $X$ is
called the **domain**, $Y$ the **codomain**, and the set of outputs actually produced,
$\{f(x) \mid x \in X\}$, is the **range** (or image).

### Three properties every ML practitioner should recognize on sight

- **Injective (one-to-one):** different inputs always give different outputs.
  $$f(x_1) = f(x_2) \implies x_1 = x_2$$
  An injective function never "collides" two distinct inputs into the same output.

- **Surjective (onto):** every element of the codomain is hit by *some* input.
  $$\forall y \in Y,\ \exists x \in X \text{ such that } f(x) = y$$

- **Bijective:** both injective *and* surjective. A bijection has a well-defined **inverse**
  function $f^{-1}: Y \to X$ that perfectly undoes $f$, i.e. $f^{-1}(f(x)) = x$ for all $x$.

A function that is *not* injective loses information — two different inputs become
indistinguishable after applying $f$. This single idea explains a lot of ML behavior, as
we'll see below.

### Composition

Given $f: X \to Y$ and $g: Y \to Z$, their composition $(g \circ f): X \to Z$ is defined by
$$(g \circ f)(x) = g(f(x))$$
— apply $f$ first, then feed the result into $g$. Composition is how simple functions build
complex ones: a deep neural network is nothing but a long composition
$f_L \circ f_{L-1} \circ \cdots \circ f_1$ of per-layer functions.

### Inverse functions

$f^{-1}$ exists (as a true function) only when $f$ is bijective. If $f$ is merely injective
but not surjective, you can still invert it on its actual range — this partial-inverse idea
is exactly why decoder architectures can only reconstruct what the encoder didn't already
throw away.

---

## 👁️ Visual Intuition

Plotting a function next to its "collision diagram" makes injectivity visual: draw a
horizontal line at any height $y$ — if it crosses the curve more than once, the function is
not injective at that height.

In [ ]:
# The horizontal-line test for injectivity, shown for three activation-like functions
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

x = np.linspace(-3, 3, 400)

funcs = {
    "Injective: $f(x) = x^3$": x**3,
    "Not injective: $f(x) = x^2$": x**2,
    "Not injective: ReLU $f(x)=\\max(0,x)$": np.maximum(0, x),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
# A per-function test height: x^3 and x^2 are tested at y=1.5 (where x^2 already shows two
# crossings, at x=+-sqrt(1.5)); ReLU is tested at y=0 instead, since ReLU's non-injective
# region is EXACTLY {x <= 0} -> 0 -- a line at any height h>0 crosses ReLU's graph only
# once (at x=h) and would misleadingly look injective, the opposite of what this demo claims.
test_heights = {
    "Injective: $f(x) = x^3$": 1.5,
    "Not injective: $f(x) = x^2$": 1.5,
    "Not injective: ReLU $f(x)=\\max(0,x)$": 0.0,
}
for ax, (title, y) in zip(axes, funcs.items()):
    test_height = test_heights[title]
    ax.plot(x, y, color="#4C72B0", lw=2)
    ax.axhline(test_height, color="#DD8452", ls="--", label=f"y = {test_height}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Notice: the horizontal line crosses x^3 once (injective), crosses x^2 TWICE at")
print("y=1.5 (not injective), and lies ALONG ReLU's entire flat region at y=0 -- every")
print("x <= 0 maps to that same output, an infinite-crossing signature of information loss.")

## 🐍 Implementation from Scratch

For *finite* mappings (like a dictionary or a lookup table), injectivity and surjectivity are
computational, not just theoretical — we can check them directly by counting collisions.

In [ ]:
# Checking injectivity/surjectivity computationally for finite mappings
def is_injective(domain, f):
    outputs = [f(x) for x in domain]
    return len(outputs) == len(set(outputs))   # no two inputs may share an output

def is_surjective(domain, codomain, f):
    outputs = {f(x) for x in domain}
    return outputs == set(codomain)            # every codomain element must be hit

domain = [-2, -1, 0, 1, 2]

square = lambda x: x ** 2
cube = lambda x: x ** 3

print("f(x) = x^2 on {-2..2}")
print("  injective? ", is_injective(domain, square), " (x=-1 and x=1 both map to 1)")
print("f(x) = x^3 on {-2..2}")
print("  injective? ", is_injective(domain, cube))

# Composition of functions, verified pointwise
f = lambda x: x + 1          # X -> Y
g = lambda y: y ** 2          # Y -> Z
g_after_f = lambda x: g(f(x))  # (g o f): X -> Z

for x in domain:
    assert g_after_f(x) == g(f(x))
print("\nComposition (g o f)(x) = g(f(x)) verified for all test inputs.")
print("(g o f)(3) =", g_after_f(3), "  step-by-step: f(3)=", f(3), "-> g(4)=", g(4))

## 🤖 Why This Matters for AI

**Activation functions.** ReLU, $f(x) = \max(0, x)$, is deliberately *not* injective — every
negative input collapses to exactly 0. This is a feature, not a bug: it creates sparsity. But
it also means gradients through "dead" ReLU units carry no information about which negative
input actually arrived — a direct consequence of losing injectivity.

**Encoder-decoder architectures.** An encoder $f_{\text{enc}}: X \to Z$ compresses inputs into
a latent space $Z$ (often lower-dimensional, so $f_{\text{enc}}$ *cannot* be injective — some
information must be lost). A decoder $f_{\text{dec}}: Z \to X$ tries to approximately invert
it. Autoencoders are literally trained to make $f_{\text{dec}} \circ f_{\text{enc}}$ as close
to the identity function as possible.

**Embeddings as functions.** A tokenizer + embedding lookup is a function
$\text{embed}: \text{Vocabulary} \to \mathbb{R}^d$. It is injective by construction (every
distinct token gets its own row of the embedding matrix) but the *learned geometry* of its
range is where all the semantic magic happens.

In [ ]:
# An autoencoder's encode-decode round trip is literally f_dec(f_enc(x)) ~ identity
import numpy as np

rng = np.random.default_rng(0)

d_in, d_latent = 8, 3     # compress 8 dims down to 3 -- encoder cannot be injective here
W_enc = rng.standard_normal((d_latent, d_in)) * 0.5
W_dec = rng.standard_normal((d_in, d_latent)) * 0.5

def encode(x):
    return W_enc @ x   # f_enc: R^8 -> R^3  (necessarily loses information: 8 > 3)

def decode(z):
    return W_dec @ z   # f_dec: R^3 -> R^8  (approximate inverse, not a true inverse)

x = rng.standard_normal(d_in)
z = encode(x)
x_reconstructed = decode(z)

print("Original x        :", np.round(x, 2))
print("Reconstructed x    :", np.round(x_reconstructed, 2))
print("Reconstruction error (L2):", round(np.linalg.norm(x - x_reconstructed), 3))
print("\nThis error is unavoidable with random weights: f_enc collapses R^8 into R^3,")
print("so it cannot be injective, and no f_dec can perfectly undo that information loss.")
print("Training an autoencoder is precisely the process of learning W_enc, W_dec that")
print("minimize this reconstruction error for the specific data distribution you care about.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x) = 2x + 3$, compute $f(0), f(1), f(-2)$ by hand, then compute $f^{-1}(3), f^{-1}(5),
   f^{-1}(-1)$ by hand using $f^{-1}(y) = (y-3)/2$. Verify $f^{-1}(f(x)) = x$ holds for all three
   original $x$ values.

<details>
<summary>💡 Solution</summary>

$f(0)=3$, $f(1)=5$, $f(-2)=-1$. Using $f^{-1}(y)=(y-3)/2$: $f^{-1}(3)=0$, $f^{-1}(5)=1$,
$f^{-1}(-1)=-2$ — each recovers exactly the $x$ that produced it, confirming
$f^{-1}(f(x))=x$ for $x\in\{0,1,-2\}$.

</details>

### 💭 UNDERSTAND
2. $f(x) = x \bmod 5$ on domain $\{0,1,\ldots,9\}$ with codomain $\{0,1,2,3,4\}$ is
   surjective but not injective. Explain in one or two sentences, without computing anything,
   why a function from a *larger* finite domain onto a *smaller* codomain can never be
   injective (this is the pigeonhole principle) — and separately, why it being surjective here
   specifically requires every remainder class $0,\ldots,4$ to actually appear among
   $0,\ldots,9$.

<details>
<summary>💡 Solution</summary>

If the domain has 10 elements and the codomain has only 5, at least two different domain
elements must share the same output the moment every output is used at all — you cannot place
10 distinct "pigeons" into 5 "holes" without at least one hole getting more than one, which is
exactly a failure of injectivity (two distinct inputs mapping to the same output). Surjectivity
here isn't automatic just from the domain being bigger, though — it holds because
$\{0,\ldots,9\}$ happens to contain at least one representative of every residue class mod 5
(in fact exactly two of each): if the domain had instead been $\{0,1,2\}$, $x\bmod 5$ would
miss outputs $3$ and $4$ entirely and would NOT be surjective onto $\{0,\ldots,4\}$.

</details>

### ✏️ DERIVE
3. Show algebraically that $f(x) = 2x+3$ is bijective on $\mathbb{R}$ (both injective and
   surjective), and derive its inverse $f^{-1}$ from scratch by solving $y = 2x+3$ for $x$ —
   don't just verify the formula given in the Warm-up above, produce it.

<details>
<summary>💡 Solution outline</summary>

**Injective:** suppose $f(x_1)=f(x_2)$, i.e. $2x_1+3 = 2x_2+3$. Subtract 3 from both sides:
$2x_1 = 2x_2$. Divide by 2: $x_1=x_2$. So equal outputs force equal inputs — injective, by the
formal definition.

**Surjective:** given any $y\in\mathbb{R}$, we need some $x$ with $f(x)=y$. Solve $y=2x+3$ for
$x$: subtract 3 ($y-3=2x$), divide by 2 ($x=(y-3)/2$). This $x$ is a real number for every real
$y$ (no division by zero, no domain restriction), so every $y$ is hit — surjective.

Since both hold, $f$ is bijective, and the inverse is exactly the formula just derived while
proving surjectivity: $f^{-1}(y) = (y-3)/2$.

</details>

### 🐍 IMPLEMENT
4. Write `is_injective`/`is_surjective` checks (reusing the notebook's own functions) for
   $f(x) = x \bmod 5$ on domain $\{0,1,\ldots,9\}$ with codomain $\{0,1,2,3,4\}$.

<details>
<summary>✅ How to know you're right</summary>

`is_injective(range(10), lambda x: x % 5)` should return `False` (e.g. both `0` and `5` map to
`0`), and `is_surjective(range(10), range(5), lambda x: x % 5)` should return `True` (every
value `0`-`4` is hit — in fact by exactly two inputs each). If you get the opposite of either
result, double check you didn't swap the domain/codomain arguments.

</details>

### 🤖 APPLY
5. A tokenizer's vocabulary lookup $\text{embed}: \text{Vocabulary} \to \mathbb{R}^d$ is
   injective by construction (the notebook's own claim in "Why This Matters for AI"). Suppose
   instead a *hashing*-based embedding scheme maps each token to `hash(token) % vocab_size`
   before lookup (a real technique used to bound vocabulary size, sometimes called the "hashing
   trick"). Implement a small toy version with a deliberately small `vocab_size` (say 8) and a
   list of 15+ distinct token strings, and check whether the resulting token-to-slot function is
   still injective.

<details>
<summary>✅ What you should observe</summary>

With more distinct tokens than `vocab_size` slots, the hashing scheme cannot be injective (same
pigeonhole argument as UNDERSTAND above) — you should find at least one collision, i.e. two
different token strings landing in the same slot, checkable the same way `is_injective` does
(comparing the count of outputs to the count of unique outputs). This is the real tradeoff the
hashing trick makes: bounded memory in exchange for occasional, unavoidable collisions between
unrelated tokens.

</details>

### 🚀 CHALLENGE
6. The softmax function $\mathbb{R}^K \to \mathbb{R}^K$ is NOT injective: `softmax(z)` and
   `softmax(z + c)` (adding the same constant `c` to every entry) give identical outputs. Verify
   this numerically for a random vector and a few values of `c`. Then explain why this
   non-injectivity is exactly what allows the numerically-stable softmax trick (subtracting
   `max(z)`) from `00.01` to work without changing the result, and identify precisely *which*
   set of inputs get mapped to the same output as a given $z$ (describe it in set-builder
   notation using the shift-invariance property, connecting back to `00.02`'s notation).

<details>
<summary>🔍 What a strong answer covers</summary>

A correct numeric check computes `softmax(z)` and `softmax(z + c)` for several random `z` and
several `c` (including negative `c`) and confirms `np.allclose` holds every time. The
explanation should state the algebraic reason directly: $\text{softmax}(z)_i =
e^{z_i}/\sum_j e^{z_j}$, and shifting every $z_i$ by $c$ multiplies both numerator and
denominator by the same factor $e^c$, which cancels in the ratio — so subtracting `max(z)`
(one particular, numerically convenient choice of shift) changes nothing about the mathematical
output, only the numerical values that get exponentiated along the way, avoiding overflow. The
strongest answers give the collision set precisely: for a fixed $z\in\mathbb{R}^K$, the full
set of inputs producing the same softmax output is $\{z + c\mathbf{1} \mid c\in\mathbb{R}\}$
(the entire line through $z$ in the direction of the all-ones vector $\mathbf{1}$) — an
*infinite* set of colliding inputs for every single output, a much stronger failure of
injectivity than the finite collisions seen in the mod-5 or hashing examples above.

---

## 📚 Further Reading
- Lay, Lay & McDonald, *Linear Algebra and Its Applications*, Appendix on functions
- [3Blue1Brown: Inverse functions](https://www.3blue1brown.com/) for visual intuition

---

*Next notebook: [The NumPy + SymPy Toolkit](04_numpy_sympy_toolkit.ipynb)*